<a href="https://colab.research.google.com/github/Davidson-Open-University/phy107-labs/blob/main/notebooks/PHY107-Lab8-Static-and-Kinetic-Friction.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PHY 107 Lab 8: Static and Kinetic Friction

**Unit 8: Friction and wear** · General Practical Physics I · Davidson Open University Global

**In this lab you will:**
- calculate coefficients of static and kinetic friction;
- find μ from the angle at which an object starts to slip;
- find μ from the gradient of a friction–normal force graph;
- predict friction forces on everyday objects.

**Time:** about 60 minutes in four parts:
- **Part A – Explore** (about 10 minutes): run the online lab and the examples, and read the explanations.
- **Part B – Practise** (about 20 minutes): fill in the blanks. A self-check tells you if you are right.
- **Part C – Build a tool** (about 20 minutes): complete a short analysis function and test it on data.
- **Part D – Your own data and reflection** (about 10 minutes): analyse readings you took in the online lab (or at home), and write short answers.

**How to use this notebook**
1. Run each code cell in order: click it and press **Shift + Enter** (or the ▶ button).
2. Where you see `...`, replace it with your own code, run the cell, then run the **check** cell under it.
   - ✅ means correct. ❌ gives you a hint: try again. ⏳ means you have not filled in the blank yet.
   - The checks never show the answer, so you learn by working it out. You can try as many times as you like.
3. Stuck? Read the hint, look back at the study session, or ask in the discussion forum. Do not skip ahead: each part builds on the last.
4. Save your work: in Colab choose **File → Save a copy in GitHub** and save it to your own `phy107-labs` repository. Offline in Jupyter, press **Ctrl + S** and later upload the file with **Add file → Upload files** on GitHub.

**Honest data:** keep your own readings separate from the example data, and label every simulated or supplied reading as such. Never invent readings.

*This lab is practice to help you learn and prepare for the unit assignment (lab report). It is not graded unless your tutor tells you otherwise.*

In [ ]:
#@title ▶ Run this cell first: it sets up the self-checks (you do not need to read it)
# You do not need to read or change this code.
import hashlib as _h
class _Blank(Exception): pass
def _norm(x, unordered=False, dp=6):
    try:
        import numpy as _np
        if isinstance(x, _np.ndarray): x = x.tolist()
        elif isinstance(x, _np.bool_): x = bool(x)
        elif isinstance(x, _np.integer): x = int(x)
        elif isinstance(x, _np.floating): x = float(x)
    except ImportError:
        pass
    if x is Ellipsis or x is None: raise _Blank()
    if isinstance(x, bool): return 'b:' + str(x)
    if isinstance(x, (int, float)): return 'n:' + repr(round(float(x), dp) + 0.0)
    if isinstance(x, str): return 't:' + ' '.join(x.lower().split())
    if isinstance(x, (set, frozenset)): return 's{' + ','.join(sorted(_norm(i, False, dp) for i in x)) + '}'
    if isinstance(x, (list, tuple)):
        items = [_norm(i, False, dp) for i in x]
        if unordered: items.sort()
        return 'l[' + ','.join(items) + ']'
    if isinstance(x, dict): return 'd{' + ','.join(sorted(_norm(k, False, dp) + '=' + _norm(v, False, dp) for k, v in x.items())) + '}'
    try:
        import sympy as _sp
        if isinstance(x, _sp.Basic):
            x = _sp.sympify(x)
            if x.is_number: return 'n:' + repr(round(float(x), dp) + 0.0)
            syms = sorted(x.free_symbols, key=str)
            vals = []
            for t in (0.3, 0.7, 1.3):
                v = complex(x.subs({s: t + 0.11 * i for i, s in enumerate(syms)}).evalf())
                vals.append(round(v.real, 6) + 0.0)
            return 'e:' + repr(vals)
    except ImportError:
        pass
    return 'r:' + repr(x)
def _hash(q, s): return _h.sha256(('phy107|' + q + '|' + s).encode()).hexdigest()[:16]
def check(q, value):
    """Check your answer to question q. Prints ✅, ❌ with a hint, or ⏳ if the blank is not filled yet."""
    spec = _CHECKS[q]
    try:
        if callable(value) and 'args' in spec:
            outs = []
            for a in spec['args']:
                try:
                    outs.append(value(*a))
                except _Blank:
                    raise
                except Exception as e:
                    print(f'❌ Question {q}: your function stopped with an error: {type(e).__name__}: {e}')
                    return
            if all(o is None or o is Ellipsis for o in outs): raise _Blank()
            got = _norm(outs, False, spec.get('dp', 6))
        elif spec.get('logic'):
            if value is None or value is Ellipsis: raise _Blank()
            import sympy as _sp
            from itertools import product as _p
            syms = sorted(_sp.sympify(value).free_symbols, key=str)
            got = 'L' + ''.join('1' if bool(_sp.sympify(value).subs(dict(zip(syms, v)))) else '0' for v in _p([True, False], repeat=len(syms))) + ':' + ','.join(map(str, syms))
        else:
            got = _norm(value, spec.get('unordered', False), spec.get('dp', 6))
    except _Blank:
        print(f'⏳ Question {q}: fill in the blank (replace ...) and run the cell above, then run this cell again.')
        return
    if _RECORD is not None:
        _RECORD[q] = _hash(q, got); print(f'✅ Question {q}: recorded.'); return
    if _hash(q, got) == spec['h']:
        print(f'✅ Question {q}: correct! ' + spec.get('ok', 'Well done.'))
    else:
        print(f'❌ Question {q}: not yet. Hint: ' + spec['hint'])
_RECORD = None

_CHECKS = {
    '8.1': {'h': 'b120d50ae5e0f55f', 'hint': 'μₛ = F / N.', 'dp': 2},
    '8.2': {'h': 'c2d03c7d202049ae', 'hint': 'μₖ = F / N while sliding at steady speed.', 'ok': 'Kinetic friction is usually less than maximum static friction.', 'dp': 2},
    '8.3': {'h': 'd1bef718c4286f8d', 'hint': 'μₛ = tan θ; use np.tan(np.radians(21.8)).', 'dp': 2},
    '8.4': {'h': '578ce7b27c585cd2', 'hint': 'np.polyfit(normal, friction, 1)[0]', 'dp': 2},
    '8.5': {'h': '41050ffd3b6ef751', 'hint': 'N = mg on a level floor, then F = μ N.', 'dp': 2},
    '8.6': {'h': '3659d3ad419a8258', 'hint': 'theta = np.arcsin(h / L); return round(float(np.tan(theta)), 3).', 'ok': 'Now you can measure μ with a book and a ruler.', 'dp': 3, 'args': [(0.30, 0.80), (0.20, 1.00), (0.50, 0.70)]},
}
print('Self-checks ready. Now carry on with Part A.')

## Part A – Explore (about 10 minutes)
### 🔬 Online lab for this unit (free open educational resources)
- **[PhET Forces and Motion: Basics (Friction screen)](https://phet.colorado.edu/en/simulations/forces-and-motion-basics)** – push objects on surfaces with different friction and read the friction force.
- **[PhET Friction](https://phet.colorado.edu/en/simulations/friction)** – see friction at the level of atoms and how rubbing warms surfaces.

Open the online lab in a new tab, try it for a few minutes, then come back here. **Label every reading you take there as "simulated".**

**Key results.** Static friction can rise up to a maximum: F_s(max) = μₛN. Once sliding, kinetic friction is F_k = μₖN, usually smaller. On a slope, an object just starts to slip when tan θ = μₛ.

**Safe home task:** put a book on a smooth board and slowly raise one end until the book starts to slide. Measure the height h and the board length L along the slope; then sin θ = h/L. A phone inclinometer app also works.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
N = 40.0
for mu in [0.2, 0.5]:
    print(f'μ = {mu}: maximum friction = {mu * N:.1f} N')

## Part B – Practise (about 20 minutes)
**B1.** A box needs 12 N to start moving when the normal force is 40 N. Find μₛ.

In [ ]:
answer_8_1 = ...

In [ ]:
check('8.1', answer_8_1)

In [ ]:
answer_8_2 = ...   # once sliding, 8.0 N keeps it moving steadily: μk

In [ ]:
check('8.2', answer_8_2)

In [ ]:
answer_8_3 = ...   # a block starts to slip at 21.8°: μs to 2 decimal places

In [ ]:
check('8.3', answer_8_3)

**B4. μₖ from a graph.** The illustrative data below give the sliding friction for different normal forces. Find μₖ as the gradient of friction against normal force (2 decimal places).

In [ ]:
normal = np.array([10, 20, 30, 40, 50])                 # N  (illustrative)
friction = np.array([2.6, 5.0, 7.4, 10.1, 12.4])        # N
plt.plot(normal, friction, 'o'); plt.xlabel('normal force (N)'); plt.ylabel('friction (N)'); plt.show()

In [ ]:
answer_8_4 = ...

In [ ]:
check('8.4', answer_8_4)

In [ ]:
answer_8_5 = ...   # kinetic friction (N) on a 5.0 kg box on a level floor with μk = 0.25

In [ ]:
check('8.5', answer_8_5)

## Part C – Build a tool (about 20 minutes)
Complete `mu_from_slope(h, L)`. For a board of length L raised to height h when slipping starts, sin θ = h/L. Return μₛ = tan θ rounded to 3 decimal places. *Hint:* `np.arcsin`.

In [ ]:
def mu_from_slope(h, L):
    ...      # YOUR CODE

In [ ]:
check('8.6', mu_from_slope)

## Part D – Your own data and reflection (about 10 minutes)
**D1. Prepares you for the Unit 8 assignment.** Do the book-on-a-board test three times for two surfaces (for example bare board and paper-covered board), or take readings in the PhET Friction screen. Find μₛ for each with your tool. Label home and simulated data.

In [ ]:
my_h = [...]     # metres
my_L = ...       # metres

**D2. Reflect.**
1. Why is the starting force usually bigger than the force to keep sliding?
2. Why do your repeated slip angles differ a little?
3. Give one example where friction is useful and one where it causes wear.

*Type your answer here (double-click this cell to edit it).*

## Finished? Save your work
In Colab: **File → Save a copy in GitHub** → your `phy107-labs` repository → commit message *"Lab 8 completed"*.
Offline: save with **Ctrl + S**, then upload the file to your repository with **Add file → Upload files** when you are online.

Then go back to Moodle and mark the lab as done.